# 2362. Generate the Invoice

## Question
We need to generate the details of the invoice with the **highest total price**.  
- If multiple invoices have the same highest price, return the one with the **smallest invoice_id**.  
- The result should show `product_id`, `quantity`, and the total `price` for each product in that invoice.

---

## Schema

### Table: Products
| Column Name | Type | Description                        |
|-------------|------|------------------------------------|
| product_id  | INT  | Primary key, ID of the product     |
| price       | INT  | Price of one unit of the product   |

---

### Table: Purchases
| Column Name | Type | Description                                      |
|-------------|------|--------------------------------------------------|
| invoice_id  | INT  | Invoice identifier                               |
| product_id  | INT  | Foreign key referencing Products                 |
| quantity    | INT  | Quantity ordered for the product in the invoice  |

**Primary Key:** (invoice_id, product_id)

---

## Sample Data

### Products
| product_id | price |
|------------|-------|
| 1          | 100   |
| 2          | 200   |

### Purchases
| invoice_id | product_id | quantity |
|------------|------------|----------|
| 1          | 1          | 2        |
| 3          | 2          | 1        |
| 2          | 2          | 3        |
| 2          | 1          | 4        |
| 4          | 1          | 10       |

---

## PySpark Code to Create Schema, Data, and Temp Views

```python
from pyspark.sql.types import StructType, StructField, IntegerType

# Schema for Products
products_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("price", IntegerType(), False)
])

# Schema for Purchases
purchases_schema = StructType([
    StructField("invoice_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("quantity", IntegerType(), False)
])

# Data for Products
products_data = [
    (1, 100),
    (2, 200)
]

# Data for Purchases
purchases_data = [
    (1, 1, 2),
    (3, 2, 1),
    (2, 2, 3),
    (2, 1, 4),
    (4, 1, 10)
]

# Create DataFrames
products_df = spark.createDataFrame(products_data, products_schema)
purchases_df = spark.createDataFrame(purchases_data, purchases_schema)

# Register Temp Views
products_df.createOrReplaceTempView("Products")
purchases_df.createOrReplaceTempView("Purchases")

# Quick check
products_df.show()
purchases_df.show()


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType

# Schema for Products
products_schema = StructType([
    StructField("product_id", IntegerType(), False),
    StructField("price", IntegerType(), False)
])

# Schema for Purchases
purchases_schema = StructType([
    StructField("invoice_id", IntegerType(), False),
    StructField("product_id", IntegerType(), False),
    StructField("quantity", IntegerType(), False)
])

# Data for Products
products_data = [
    (1, 100),
    (2, 200)
]

# Data for Purchases
purchases_data = [
    (1, 1, 2),
    (3, 2, 1),
    (2, 2, 3),
    (2, 1, 4),
    (4, 1, 10)
]

# Create DataFrames
products_df = spark.createDataFrame(products_data, products_schema)
purchases_df = spark.createDataFrame(purchases_data, purchases_schema)

# Register Temp Views
products_df.createOrReplaceTempView("Products")
purchases_df.createOrReplaceTempView("Purchases")

# Quick check
products_df.show()
purchases_df.show()

In [0]:
%sql
with cte as (
  Select pur.invoice_id as purchase_invoice_id , pur.product_id as purchase_product_id  , pur.quantity as purchase_quantity  , pdct.product_id ,pdct.price  as product_price , (pur.quantity * pdct.price ) as bill
  from Products  as pdct 
  left join Purchases  pur
  on pur.product_id = pdct.product_id
) , cte2 as (
Select purchase_product_id , purchase_invoice_id , bill , purchase_quantity , product_price
 ,sum(purchase_quantity)over(partition by purchase_invoice_id ) as total_quantity
 , sum(cte.bill)over(partition by purchase_invoice_id) as total_bill 
--, row_number()over(partition by  purchase_product_id order by  purchase_invoice_id asc  ) 
 from cte 

)
,cte3 as (
 select rank()over(partition by  total_bill order by purchase_invoice_id  ) as rn_1  ,
RANK() Over(order by  total_bill DESC ) AS RNK,
 purchase_product_id  , purchase_quantity ,bill , total_bill , purchase_invoice_id from cte2  
)
Select purchase_product_id  as product_id  ,purchase_quantity as quantity  ,bill as price  from cte3
 WHERE rn_1 = 1 and rnk = 1

In [0]:
%sql
with cte_4 as (
  Select  pur.invoice_id as purchase_invoice_id , sum(pur.quantity * pdct.price ) as bill
  from Products  as pdct 
  left join Purchases  pur
  on pur.product_id = pdct.product_id
  group by pur.invoice_id
  order by  bill desc ,  pur.invoice_id asc
  limit 1 )
  select   pdct.product_id as product_id , pur.quantity as quantity   , (pur.quantity * pdct.price ) as price 
  from Products  as pdct 
  left join Purchases  pur
  on pur.product_id = pdct.product_id
  inner join  cte_4 on pur.invoice_id = cte_4.purchase_invoice_id

# Reflection on Problem-Solving

## Time Taken
- **1 hour 40 minutes**

## Why?
- I did not read the question carefully at first.
- Misunderstanding the requirements led to unnecessary complexity.

## What I Understood
- Importance of reading the question thoroughly.
- Take time to understand the problem before jumping into coding.

## What I Learned
- Initially, I tried solving with **3 CTEs**.
- Later, I realized the solution could be simplified to **just 1 CTE**.
- Careful analysis reduces unnecessary complexity.

---

# Code Comparison

### Approach 1: Multiple CTEs

**Step-by-step**

**cte**  
- Joins `Products` and `Purchases`.  
- Calculates the line item bill = `quantity * price`.  
- Produces detailed rows per product per invoice.  

**cte2**  
- Adds window functions:  
  - `total_quantity` → total items in each invoice.  
  - `total_bill` → total invoice amount.  
- Still keeps product-level detail.  

**cte3**  
- Uses `RANK()` twice:  
  - `rn_1` → ensures we pick the first product row per invoice (avoid duplicates).  
  - `rnk` → ranks invoices by total bill (highest first).  
- This isolates the invoice with the maximum bill.  

**Final SELECT**  
- Filters to the invoice with `rnk = 1` (highest bill).  
- Ensures only one invoice is chosen (`rn_1 = 1`).  
- Returns product details for that invoice.  

---

### Approach 2: Simplified Single CTE

**Step-by-step**

**cte_4**  
- Aggregates invoice totals using `SUM(quantity * price)`.  
- Orders invoices by total bill (descending) and `invoice_id` (ascending).  
- Picks the highest bill invoice (with smallest ID if tie).  

**Final SELECT**  
- Joins back to `Products` and `Purchases`.  
- Filters only the chosen invoice from `cte_4`.  
- Returns product details (`product_id`, `quantity`, line item price).  

---

## Key Takeaways
- **Version 1**: More complex, uses multiple CTEs and window functions. Good for practicing ranking logic, but over-engineered.  
- **Version 2**: Cleaner, uses aggregation + `LIMIT 1`. Easier to read and more efficient.  
- Lesson: **Read carefully, think clearly, and aim for simplicity.**
